In [ ]:
! pip install --upgrade xarray zarr gcsfs cftime nc-time-axis

In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import gcsfs
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = 12, 6
gcs = gcsfs.GCSFileSystem(token="anon")

# catalog of every dataset in the google cloud cmip6 collection
catalog = pd.read_csv("https://storage.googleapis.com/cmip6/cmip6-zarr-consolidated-stores.csv")
catalog.head()

,activity_id,institution_id,source_id,experiment_id,member_id,table_id,variable_id,grid_label,zstore,dcpp_init_year,version
0,HighResMIP,CMCC,CMCC-CM2-HR4,highresSST-present,r1i1p1f1,Amon,ps,gn,gs://cmip6/CMIP6/HighResMIP/CMCC/CMCC-CM2-HR4/...,NaN,20170706
1,HighResMIP,CMCC,CMCC-CM2-HR4,highresSST-present,r1i1p1f1,Amon,rsds,gn,gs://cmip6/CMIP6/HighResMIP/CMCC/CMCC-CM2-HR4/...,NaN,20170706
2,HighResMIP,CMCC,CMCC-CM2-HR4,highresSST-present,r1i1p1f1,Amon,rlus,gn,gs://cmip6/CMIP6/HighResMIP/CMCC/CMCC-CM2-HR4/...,NaN,20170706
3,HighResMIP,CMCC,CMCC-CM2-HR4,highresSST-present,r1i1p1f1,Amon,rlds,gn,gs://cmip6/CMIP6/HighResMIP/CMCC/CMCC-CM2-HR4/...,NaN,20170706
4,HighResMIP,CMCC,CMCC-CM2-HR4,highresSST-present,r1i1p1f1,Amon,psl,gn,gs://cmip6/CMIP6/HighResMIP/CMCC/CMCC-CM2-HR4/...,NaN,20170706


In [2]:
# which experiments exist, looking for historical, ssp126, ssp245, ssp585
catalog.experiment_id.value_counts().head(20)

experiment_id
dcppA-hindcast     141016
historical          86693
ssp245              39097
ssp585              34643
ssp370              32739
ssp126              27676
abrupt-4xCO2        12694
1pctCO2             12042
piControl           10834
ssp119              10157
amip                 7949
pdSST-pdSIC          6275
pdSST-futArcSIC      5774
hist-GHG             5244
hist-nat             4981
esm-hist             4902
hist-aer             4208
ssp534-over          3530
pdSST-piArcSIC       3253
ssp434               3223
Name: count, dtype: int64

In [3]:
# which monthly variables exist, looking for tas (air temperature) and pr (precipitation)
catalog.query("table_id == 'Amon'").variable_id.value_counts().head(20)

variable_id
tas        15492
psl        12938
clt        11467
ps         11378
evspsbl    11080
ts         10537
rsdt        8365
huss        8108
rsds        7460
rlut        7409
hus         7306
ta          7230
ua          7176
va          7172
hfls        7120
zg          7033
pr          6861
hfss        6850
rsus        6770
tasmax      6699
Name: count, dtype: int64

In [4]:
# which models have monthly temperature for the high emissions scenario
catalog.query("variable_id == 'tas' & table_id == 'Amon' & experiment_id == 'ssp585'").source_id.unique()

array(['GFDL-CM4', 'GFDL-ESM4', 'CNRM-CM6-1', 'BCC-CSM2-MR',
       'CNRM-ESM2-1', 'CanESM5', 'CanESM5-CanOE', 'UKESM1-0-LL',
       'AWI-CM-1-1-MR', 'INM-CM4-8', 'MIROC6', 'CAMS-CSM1-0',
       'MPI-ESM1-2-LR', 'MPI-ESM1-2-HR', 'INM-CM5-0', 'NESM3',
       'MCM-UA-1-0', 'FGOALS-g3', 'MIROC-ES2L', 'IPSL-CM6A-LR',
       'KACE-1-0-G', 'FGOALS-f3-L', 'MRI-ESM2-0', 'NorESM2-LM',
       'NorESM2-MM', 'CNRM-CM6-1-HR', 'FIO-ESM-2-0', 'HadGEM3-GC31-LL',
       'GISS-E2-1-G', 'GISS-E2-1-H', 'EC-Earth3-Veg', 'EC-Earth3',
       'CESM2-WACCM', 'ACCESS-CM2', 'HadGEM3-GC31-MM', 'CESM2', 'CIESM',
       'CMCC-CM2-SR5', 'TaiESM1', 'IITM-ESM', 'E3SM-1-1',
       'EC-Earth3-Veg-LR', 'CAS-ESM2-0', 'EC-Earth3-CC', 'CMCC-ESM2',
       'ACCESS-ESM1-5', 'KIOST-ESM'], dtype=object)

In [5]:
# gfdl-esm4 has historical plus all three future scenarios for this run
model = "GFDL-ESM4"
member = "r1i1p1f1"
names = {"ssp126": "low (SSP1-2.6)", "ssp245": "middle (SSP2-4.5)", "ssp585": "high (SSP5-8.5)"}

def load(var, exp):
    # use the newest version if there are several
    row = catalog.query(
        "table_id == 'Amon' & variable_id == @var & experiment_id == @exp "
        "& source_id == @model & member_id == @member"
    ).sort_values("version").iloc[-1]
    da = xr.open_zarr(gcs.get_mapper(row.zstore), consolidated=True)[var]
    # switch longitude from 0 to 360 into -180 to 180 so it lines up with normal maps
    return da.assign_coords(lon=((da.lon + 180) % 360) - 180).sortby("lon")

def yearly_global(da):
    # cells near the poles cover less area, so weight by cos(latitude)
    w = np.cos(np.deg2rad(da.lat))
    return da.weighted(w).mean(["lat", "lon"]).groupby("time.year").mean()

In [6]:
# global warming per year, measured against the 1850 to 1900 average
# this cell takes a few minutes since it downloads the data
hist = load("tas", "historical")
hist_g = yearly_global(hist).load()
base = hist_g.sel(year=slice(1850, 1900)).mean()

future = {}
future_g = {}
for s in names:
    future[s] = load("tas", s).sel(time=slice("2015", "2100"))
    future_g[s] = yearly_global(future[s]).load()

In [ ]:
# plot 1: global warming under each scenario
(hist_g - base).plot(color="black", label="historical")
for s in names:
    (future_g[s] - base).plot(label=names[s])
plt.axhline(0, color="gray", lw=0.5)
plt.title(f"change in global air temperature vs 1850-1900 ({model})")
plt.ylabel("change in air temperature (°C)")
plt.figtext(0.13, 0.01, "Each line is the same climate model run under a different emissions future", fontsize=9, color="gray")
plt.legend()
plt.show()

In [16]:
lat = np.arange(-89.5, 90, 1)
lon = np.arange(-179.5, 180, 1)

def regrid(da):
    return (da.interp(lat=lat, kwargs={"fill_value": "extrapolate"})
              .interp(lon=lon, kwargs={"fill_value": "extrapolate"}))

base_map = regrid(hist.sel(time=slice("1850", "1900")).mean("time").load())

maps = {}
for s in names:
    yr = future[s].sel(time=slice("2020", "2099")).groupby("time.year").mean()
    dec = yr.groupby((yr.year // 10 * 10).rename("decade")).mean().load()
    maps[s] = regrid(dec) - base_map

In [ ]:
# plot 2 with a slider to step through decades
from ipywidgets import interact

def show(decade):
    maps["ssp585"].sel(decade=decade).plot(cmap="Reds", vmin=0, vmax=top, cbar_kwargs={"label": "change in air temperature (°C)"})
    plt.title(f"change in air temperature in the {decade}s, high emissions")
    plt.show()

top = float(maps["ssp585"].sel(decade=2090).max())
interact(show, decade=(2020, 2090, 10))

In [ ]:
# plot 3: low vs high emissions side by side, same color scale
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
top = float(maps["ssp585"].sel(decade=2090).max())
for ax, s in zip(axes, ["ssp126", "ssp585"]):
    maps[s].sel(decade=2090).plot(ax=ax, cmap="Reds", vmin=0, vmax=top, cbar_kwargs={"label": "change in air temperature (°C)"})
    ax.set_title(f"change in air temperature, 2090s, {names[s]}")
plt.show()

In [ ]:
# plot 4: warming by latitude, to see if the poles warm faster
for s in names:
    z = maps[s].sel(decade=2090).mean("lon")
    plt.plot(z.lat, z, label=names[s])
plt.title("change in air temperature by latitude, 2090s")
plt.xlabel("latitude (south to north)")
plt.ylabel("change in air temperature (°C)")
plt.legend()
plt.show()

In [ ]:
# plot 5: san diego compared to the global average, high emissions
# note: tas is stored in kelvin, but a change in kelvin equals a change in celsius,
# so the anomalies below are already in °C and need no conversion
# 10 year rolling average to smooth out the year to year noise
sd = dict(lat=32.7, lon=-117.2, method="nearest")
sd_hist = hist.sel(**sd).groupby("time.year").mean().load()
sd_base = sd_hist.sel(year=slice(1850, 1900)).mean()
sd_high = future["ssp585"].sel(**sd).groupby("time.year").mean().load()

sd_all = xr.concat([sd_hist, sd_high], dim="year") - sd_base
global_all = xr.concat([hist_g, future_g["ssp585"]], dim="year") - base

sd_all.rolling(year=10, center=True).mean().plot(label="san diego")
global_all.rolling(year=10, center=True).mean().plot(color="gray", label="global average")
plt.axvline(2015, color="gray", lw=0.5, ls="--")
plt.title("change in air temperature: san diego vs global average, high emissions")
plt.ylabel("change in air temperature (°C)")
plt.legend()
plt.show()

In [ ]:
# plot 6: rainfall change by the 2090s under high emissions, as a percent
# brown is drier, green is wetter
pr_base = load("pr", "historical").sel(time=slice("1850", "1900")).mean("time").load()
pr_end = load("pr", "ssp585").sel(time=slice("2090", "2099")).mean("time").load()

((pr_end - pr_base) / pr_base * 100).plot(cmap="BrBG", vmin=-50, vmax=50)
plt.title("rainfall change by the 2090s, high emissions (%)")
plt.show()

In [17]:
def yearly_grid(da):
    yr = da.groupby("time.year").mean().load()
    return regrid(yr)

recent = yearly_grid(hist.sel(time=slice("1985", "2014")))

warming = {}
for s in names:
    both = xr.concat([recent, yearly_grid(future[s])], dim="year")
    # change vs 1850-1900, then a 10 year centered average so one hot year can't count as a crossing
    warming[s] = (both - base_map).rolling(year=10, center=True, min_periods=5).mean().sel(year=slice(2000, 2100))

In [18]:
# plot 7: the year each place first reaches a warming threshold
import ipywidgets as widgets
from ipywidgets import interact

cmap = plt.get_cmap("YlOrRd_r").copy()
cmap.set_bad("#bbbbbb")   # gray = never reaches the threshold by 2100

def crossing_year(a, threshold):
    # first year the smoothed warming is at or above the threshold (NaN if it never is)
    return a.year.where(a >= threshold).min("year")

def show_crossing(threshold, scenario):
    cross = crossing_year(warming[scenario], threshold)

    # share of the globe's area that reaches the threshold by 2100, weighted by cos(latitude)
    w = np.cos(np.deg2rad(cross.lat))
    share = float(cross.notnull().astype(float).weighted(w).mean()) * 100

    cross.plot(cmap=cmap, vmin=2000, vmax=2100,
               cbar_kwargs={"label": "year the threshold is first reached"})
    plt.title(f"when does each place reach {threshold}°C of warming? {names[scenario]}\n"
              f"{share:.0f}% of the globe's area gets there by 2100 (gray = never)",
              fontsize=11)
    plt.figtext(0.13, 0.01,
                "Warming is the 10 year average vs 1850-1900. A place is colored by the first year that average reaches the threshold.",
                fontsize=8, color="gray")
    plt.show()

interact(
    show_crossing,
    threshold=widgets.SelectionSlider(options=[1, 1.5, 2, 2.5, 3], value=1.5, description="threshold (°C)"),
    scenario=widgets.Dropdown(options=[(names[s], s) for s in names], value="ssp585", description="scenario"),
);

interactive(children=(SelectionSlider(description='threshold (°C)', index=1, options=(1, 1.5, 2, 2.5, 3), valu…

In [19]:
# Add as a new cell after cell 18 (needs `warming`, `names`, `lat`, `lon`, `model`, `crossing_year`).
# Writes data/crossing_data.js, which plot7.html loads with a plain <script> tag (works from file://, no server).
import json, os
import numpy as np

os.makedirs("data", exist_ok=True)
thresholds = [1, 1.5, 2, 2.5, 3]

out = {
    "model": model,
    "lat": [float(v) for v in lat],
    "lon": [float(v) for v in lon],
    "thresholds": thresholds,
    "scenarios": [],
}
for s in names:
    grids = {}
    for t in thresholds:
        c = crossing_year(warming[s], t).transpose("lat", "lon")   # same function the notebook plot uses
        # row-major (lat rows, lon columns); null = never reaches the threshold by 2100
        grids[str(t)] = [None if np.isnan(v) else int(v) for v in c.values.ravel()]
    out["scenarios"].append({"id": s, "label": names[s], "grids": grids})

with open("data/crossing_data.js", "w") as f:
    f.write("window.CROSSING = " + json.dumps(out, separators=(",", ":")) + ";")
print("wrote data/crossing_data.js")

wrote data/crossing_data.js


In [20]:
w = warming["ssp585"]
print("warming:", w.dims, w.shape)
print("lat:", w.lat.values[:3], w.lat.values[-3:])
print("lon:", w.lon.values[:3], w.lon.values[-3:])
print("NaN share in 2000:", float(w.sel(year=2000).isnull().mean()))
print("NaN share in 2100:", float(w.sel(year=2100).isnull().mean()))
print("base_map:", base_map.shape, "NaN share:", float(base_map.isnull().mean()))
print("native grid:", dict(hist.sizes))

warming: ('year', 'lat', 'lon') (101, 180, 360)
lat: [-89.5 -88.5 -87.5] [87.5 88.5 89.5]
lon: [-179.5 -178.5 -177.5] [177.5 178.5 179.5]
NaN share in 2000: 0.0
NaN share in 2100: 0.0
base_map: (180, 360) NaN share: 0.0
native grid: {'time': 1980, 'lat': 180, 'lon': 288}


In [ ]:
len(out["scenarios"][0]["grids"]["1.5"])

# plot 8: warming over time in the arctic, tropics and antarctic vs the globe, high emissions
# uses the 10 year smoothed 5 degree grid from the crossing-year cell
bands = {
    "arctic (60-90°N)": (60, 90),
    "tropics (30°S-30°N)": (-30, 30),
    "antarctic (60-90°S)": (-90, -60),
}

def area_mean(a, lo=-90, hi=90):
    b = a.sel(lat=slice(lo, hi))
    return b.weighted(np.cos(np.deg2rad(b.lat))).mean(["lat", "lon"])

w = warming["ssp585"]
for label, (lo, hi) in bands.items():
    area_mean(w, lo, hi).plot(label=label)
area_mean(w).plot(color="black", ls="--", label="global average")

plt.title("change in air temperature by region, high emissions")
plt.ylabel("change in air temperature (°C)")
plt.xlabel("year")
plt.legend()
plt.show()

# plot 9: warming by season, 2090s vs 1850-1900, high emissions
def season_mean(da, start, end):
    m = da.sel(time=slice(start, end)).groupby("time.season").mean("time").load()
    return m.interp(lat=lat, lon=lon)

season_diff = season_mean(future["ssp585"], "2090", "2099") - season_mean(hist, "1850", "1900")

top = float(season_diff.sel(season=["DJF", "JJA"]).max())
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
for ax, season, label in zip(axes, ["DJF", "JJA"], ["Dec-Feb", "Jun-Aug"]):
    season_diff.sel(season=season).plot(
        ax=ax, cmap="Reds", vmin=0, vmax=top,
        cbar_kwargs={"label": "change in air temperature (°C)"},
    )
    ax.set_title(f"change in air temperature, 2090s, {label}, high emissions")
plt.show()

# models that have monthly tas for historical + all three scenarios with the same member
need = ["historical", "ssp126", "ssp245", "ssp585"]
ok = None
for e in need:
    s = set(catalog.query("variable_id == 'tas' & table_id == 'Amon' & experiment_id == @e "
                          "& member_id == @member").source_id)
    ok = s if ok is None else ok & s
print(len(ok), sorted(ok))

# plot 8: multi-model mean and min/max spread for each scenario
colors = {"ssp126": "tab:blue", "ssp245": "tab:orange", "ssp585": "tab:red"}
fig, ax = plt.subplots()
for s in names:
    df = pd.DataFrame({m: v.to_series() for m, v in series[s].items()})
    df = df.rolling(10, center=True, min_periods=5).mean()
    ax.fill_between(df.index, df.min(axis=1), df.max(axis=1), color=colors[s], alpha=0.2)
    ax.plot(df.index, df.mean(axis=1), color=colors[s], label=names[s])
ax.axhline(1.5, color="gray", lw=0.6, ls="--")
ax.axhline(2.0, color="gray", lw=0.6, ls="--")
ax.set_title(f"global warming vs 1850-1900, multi-model mean and range ({len(models)} models)")
ax.set_ylabel("change in global air temperature (°C)")
ax.legend()
plt.show()

# plot 9: each dot is one model's warming in the 2090s
end = {s: pd.Series({m: float(v.sel(year=slice(2090, 2099)).mean()) for m, v in series[s].items()})
       for s in names}

rng = np.random.default_rng(0)
for i, s in enumerate(names):
    y = end[s]
    plt.scatter(i + rng.uniform(-0.12, 0.12, len(y)), y, color=colors[s], alpha=0.8)
    plt.hlines(y.mean(), i - 0.3, i + 0.3, color="black")
plt.xticks(range(3), [names[s] for s in names])
plt.title("warming in the 2090s: spread across models vs spread across scenarios")
plt.ylabel("change in global air temperature (°C)")
plt.show()

model_spread = max(end[s].max() - end[s].min() for s in names)
scenario_gap = end["ssp585"].mean() - end["ssp126"].mean()
print(f"biggest disagreement between models inside one scenario: {model_spread:.1f}°C")
print(f"gap between the low and high scenario means:            {scenario_gap:.1f}°C")

# plot 10: san diego warming by season, 2090s vs 1850-1900
def sd_season(da, start, end):
    return da.sel(time=slice(start, end)).groupby("time.season").mean("time").load()

sd_b = sd_season(hist.sel(**sd), "1850", "1900")
order = ["DJF", "MAM", "JJA", "SON"]
x = np.arange(4)
for i, s in enumerate(names):
    d = sd_season(future[s].sel(**sd), "2090", "2099") - sd_b
    plt.bar(x + (i - 1) * 0.27, d.sel(season=order), 0.27, color=colors[s], label=names[s])
plt.xticks(x, ["Dec-Feb", "Mar-May", "Jun-Aug", "Sep-Nov"])
plt.title("san diego: change in air temperature by season, 2090s")
plt.ylabel("change in air temperature (°C)")
plt.legend()
plt.show()

# plot 11: days per year hotter than the 1985-2014 95th percentile, san diego
# tasmax is daily, so this is a big download (several minutes)
def sd_daily(exp):
    da = load_m("tasmax", exp, model, table="day")
    return da.sel(lat=32.7, lon=360 - 117.2, method="nearest")   # this dataset uses 0-360 longitude

h = sd_daily("historical").load()
thr = float(h.sel(time=slice("1985", "2014")).quantile(0.95))

def hot_days(da):
    return (da > thr).groupby("time.year").sum()

hot_hist = hot_days(h)
plt.plot(hot_hist.year, hot_hist.rolling(year=10, center=True).mean(), color="black", label="historical")
for s in names:
    f = sd_daily(s).sel(time=slice("2015", "2100")).load()
    hd = hot_days(f)
    plt.plot(hd.year, hd.rolling(year=10, center=True).mean(), color=colors[s], label=names[s])
plt.title("san diego: hot days per year (above the 1985-2014 95th percentile)")
plt.ylabel("days per year (10 year average)")
plt.legend()
plt.show()

# plot 12: san diego annual rainfall, 20 year average because year to year swings are huge
def sd_rain(exp, start, end):
    pr = load_m("pr", exp, model).sel(time=slice(start, end))
    pr = pr.sel(lat=32.7, lon=360 - 117.2, method="nearest").groupby("time.year").mean().load()
    return pr * 86400 * 365   # kg m-2 s-1 to mm per year

r_hist = sd_rain("historical", "1850", "2014")
plt.plot(r_hist.year, r_hist.rolling(year=20, center=True).mean(), color="black", label="historical")
for s in names:
    r = sd_rain(s, "2015", "2100")
    plt.plot(r.year, r.rolling(year=20, center=True).mean(), color=colors[s], label=names[s])
plt.title("san diego: annual rainfall (20 year average)")
plt.ylabel("rainfall (mm per year)")
plt.legend()
plt.show()